# Text Detoxification - Text Style Transfer From Toxic Comment to Neutral Comment

## Install Dependencies

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install -q -U transformers accelerate datasets evaluate
!pip install sentencepiece sacremoses --quiet
!pip install nltk scikit-learn --quiet
!pip install evaluate --quiet
!pip install -q --upgrade transformers
!pip install wandb --quiet
!pip install sacrebleu --quiet


## Prepare Dataset

In [ ]:
import os, random, pickle
import pandas as pd
from datasets import load_dataset

random.seed(53)

# Load the dataset
dataset = load_dataset("s-nlp/paradetox", split="train")

# Drop entries with missing values
dataset = dataset.filter(lambda x: x['en_toxic_comment'] and x['en_neutral_comment'])

# Shuffle deterministically
dataset = dataset.shuffle(seed=53)

parallel_data = dataset.select(range(6250))

# Create (tox, detox) pairs
pairs = [{'toxic': ex['en_toxic_comment'], 'neutral': ex['en_neutral_comment']} for ex in parallel_data]

# Define splits
train_pairs     = pairs[:1000]
dev_pairs       = pairs[5000:5250]
fulltrain_pairs = pairs[:5000]
test_pairs      = pairs[6000:6500]

# Convert to DataFrames
train_df      = pd.DataFrame(train_pairs)
dev_df        = pd.DataFrame(dev_pairs)
fulltrain_df  = pd.DataFrame(fulltrain_pairs)
test_df       = pd.DataFrame(test_pairs)

# Save data
os.makedirs("/content/genai_proj_low_resource_tst/data", exist_ok=True)
train_df.to_csv("/content/genai_proj_low_resource_tst/data/train.csv", index=False)
dev_df.to_csv("/content/genai_proj_low_resource_tst/data/dev.csv", index=False)
fulltrain_df.to_csv("/content/genai_proj_low_resource_tst/data/fulltrain.csv", index=False)
test_df.to_pickle("/content/genai_proj_low_resource_tst/data/test.pkl")

print("Wrote train.csv, dev.csv, fulltrain.csv, and test.pkl")
print(f"Train size: {len(train_df)}, Dev size: {len(dev_df)}, Test size: {len(test_df)}")

# Prepare 2,000 additional toxic-only sentences for augmentation/self-training
unlabeled_toxics = dataset.select(range(6500, 8500))
toxic_lines = [ex['en_toxic_comment'] for ex in unlabeled_toxics if ex['en_toxic_comment']]
unparallel_data_df = pd.DataFrame({'toxic': toxic_lines})
unparallel_data_df.to_pickle("/content/genai_proj_low_resource_tst/data/unparallel.pkl")

print("Wrote data/unparallel.pkl")
print("Total unparallel toxic examples:", len(unparallel_data_df))


## Data Augmentation

In [ ]:
!pip install nlpaug sentencepiece transformers datasets --quiet

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


In [ ]:
import os
import random
import pandas as pd
import nlpaug.augmenter.word as naw
from tqdm import tqdm

# For reproducibility
random.seed(53)

# --- Load your data ---
train_path = "/content/genai_proj_low_resource_tst/data/train.csv"
dev_path   = "/content/genai_proj_low_resource_tst/data/dev.csv"

train_df = pd.read_csv(train_path)
dev_df   = pd.read_csv(dev_path)

# --- Define only one augmenter: SynonymAug ---
augmenter = naw.SynonymAug(aug_src='wordnet')

def augment_df(df, prob=0.5):
    augmented_rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Apply augmentation with 50% probability
        if random.random() < prob:
            new_tox = augmenter.augment(row["toxic"])
            new_detox = row["neutral"]
            augmented_rows.append({
                "toxic": new_tox,
                "neutral": new_detox,
            })
    return pd.DataFrame(augmented_rows)

# --- Run augmentation ---
aug_train_df = augment_df(train_df)
aug_dev_df   = augment_df(dev_df)

print(f"Original train: {len(train_df)}, Augmented train: {len(aug_train_df)}")
print(f"Original dev: {len(dev_df)}, Augmented dev: {len(aug_dev_df)}")

# --- Combine original + augmented ---
train_all = pd.concat([train_df, aug_train_df[["toxic", "neutral"]]], ignore_index=True)
dev_all   = pd.concat([dev_df, aug_dev_df[["toxic", "neutral"]]], ignore_index=True)

# --- Save to same folder ---
os.makedirs("/content/genai_proj_low_resource_tst/data/", exist_ok=True)
train_all.to_csv("/content/genai_proj_low_resource_tst/data/train.csv", index=False)
dev_all.to_csv("/content/genai_proj_low_resource_tst/data/dev.csv", index=False)

print("Synonym-based augmented datasets saved in /content/genai_proj_low_resource_tst/data")


## Style Transfer Train

In [ ]:
import os
import random
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

tox_texts = [p['toxic_comment'] for p in pairs]
detox_texts = [p['neutral_comment'] for p in pairs]

texts = tox_texts + detox_texts
labels = [1] * len(tox_texts) + [0] * len(detox_texts)

print(f"Total labeled samples: {len(texts)} (Pos={len(tox_texts)}, Neg={len(detox_texts)})")


train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=53, stratify=labels
)

print(f"Train size: {len(train_texts)}, Val size: {len(val_texts)}")
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")



from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_enc = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

train_dataset = SentimentDataset(train_enc, train_labels)
val_dataset = SentimentDataset(val_enc, val_labels)


from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./bert_style_classifier",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./bert_style_classifier/logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
from huggingface_hub import HfApi, create_repo
from transformers import BertForSequenceClassification, BertTokenizer

repo_name = "bert-style-classifier-detox"  =
username = "shubhamkr"

create_repo(f"{username}/{repo_name}", exist_ok=True)

# Save locally first
save_dir = f"./{repo_name}"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Then push to Hub
model.push_to_hub(f"{username}/{repo_name}")
tokenizer.push_to_hub(f"{username}/{repo_name}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "shubhamkr/bert-style-classifier-detox"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


In [ ]:
test_df = pd.read_pickle("/content/genai_proj_low_resource_tst/data/test.pkl")

pos_texts = test_df["toxic_comment"].tolist()
neg_texts = test_df["neutral_comment"].tolist()

# Combine into one list of texts + labels
texts = pos_texts + neg_texts
labels = [1]*len(pos_texts) + [0]*len(neg_texts)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 16
correct = 0
all_preds = []
all_labels = []

for i in tqdm(range(0, len(texts), batch_size), desc="Evaluating"):
    batch_texts = texts[i:i+batch_size]
    batch_labels = labels[i:i+batch_size]

    enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch_labels)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {acc*100:.2f}%\n")

print("Detailed report:")
print(classification_report(all_labels, all_preds, target_names=["neutral_comment", "toxic_comment"]))


## Evaluation Code

In [ ]:
import torch
import sys
import random
import numpy as np
import pickle
import argparse
import evaluate
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, GPT2Tokenizer, GPT2LMHeadModel, set_seed

class AutomaticEvaluator:
    def __init__(self, seed_value):
        self.seed_value = seed_value
        self.bleu = None
        self.sim_model = None
        self.detox_tokenizer = None
        self.detox_model = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.gpt2_tokenizer = None
        self.gpt2_model = None

    def set_seed(self):
        random.seed(self.seed_value)
        np.random.seed(self.seed_value)
        torch.manual_seed(self.seed_value)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.seed_value)
            torch.cuda.manual_seed_all(self.seed_value)
        set_seed(self.seed_value)

    def load_models(self):
        print("Loading evaluation models...")
        self.bleu = evaluate.load("bleu")
        self.sim_model = SentenceTransformer('bert-base-nli-mean-tokens')
        self.critic_model = None
        model_name = "shubhamkr/bert-style-classifier-detox"
        self.detox_tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.detox_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.detox_model.eval()

        # GPT-2 for fluency
        with torch.no_grad():
            self.gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
            self.gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
            self.gpt2_model.eval()

        print("Models loaded successfully.")


    def gpt2_score(self, sentence):
        tokenize_input = self.gpt2_tokenizer.encode(sentence, return_tensors='pt')
        with torch.no_grad():
            loss = self.gpt2_model(tokenize_input, labels=tokenize_input).loss
        return torch.exp(loss).item()

    def similarity(self, sentence1, sentence2):
        # Sentences can be encoded in batches for efficiency
        embeddings = self.sim_model.encode([sentence1, sentence2])
        sim_score = cosine_similarity([embeddings[0]], [embeddings[1]])
        return sim_score[0][0]

    def senti_score(self, sentence):
        inputs = self.detox_tokenizer(
            sentence,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(self.device)
        with torch.no_grad():
            logits = self.detox_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)
            pred = torch.argmax(probs, dim=-1).item()

        # Assuming label 1 = TOXIC, 0 = DETOX
        return 1 if pred == 0 else 0

    def evaluate_all(self, srcs_file, refs_file, preds_file):
        with open(srcs_file, 'rb') as f:
            sources = pickle.load(f)
        with open(refs_file, 'rb') as f:
            references = pickle.load(f)
        with open(preds_file, 'rb') as f:
            predictions = pickle.load(f)

        print(f"Evaluating {len(predictions)} predictions...")

        # Prepare data for BLEU score calculation
        references_bleu = [[ref] for ref in references]
        sources_bleu = [[src] for src in sources]

        sim_scores = []
        gpt2_scores = []
        correct_count = 0

        for i in range(len(predictions)):
            pred = predictions[i]
            ref = references[i]

            # 1. Accuracy Score
            correct_count += self.senti_score(pred)

            # 2. Similarity Score
            sim_scores.append(self.similarity(pred, ref))

            # 3. Fluency Score (Perplexity)
            gpt2_scores.append(self.gpt2_score(pred))

        # 4. BLEU Scores (calculated in batch)
        bleu_withsrc = self.bleu.compute(predictions=predictions, references=sources_bleu, max_order=4)
        bleu_withtrg = self.bleu.compute(predictions=predictions, references=references_bleu, max_order=4)

        # Calculate final averages
        acc = correct_count / len(predictions)
        avg_sim = sum(sim_scores) / len(sim_scores)
        avg_gpt2_ppl = sum(gpt2_scores) / len(gpt2_scores)

        return (
            acc,
            bleu_withsrc['bleu'],
            bleu_withtrg['bleu'],
            avg_sim,
            avg_gpt2_ppl
        )


def main(args):
    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
    )

    print('-' * 20)
    print(f"Evaluation Results for Task: {args.task}")
    print('-' * 20)
    print(f"Accuracy: {acc:.4f}")
    print(f"Bleu with Source: {bleu_withsrc:.4f}")
    print(f"Bleu with Target: {bleu_withtrg:.4f}")
    print(f"Similarity: {sim:.4f}")
    # print(f"GPT-2 PPL (Fluency): {gpt2_ppl:.4f}")


## Self Training with Style Reward Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

style_model_name = "shubhamkr/bert-style-classifier-detox"
style_tokenizer = AutoTokenizer.from_pretrained(style_model_name)
style_classifier = AutoModelForSequenceClassification.from_pretrained(style_model_name).to(device)
style_classifier.eval()


In [ ]:
from transformers import Seq2SeqTrainer
import torch


class StyleRewardTrainer(Seq2SeqTrainer):
    def __init__(self, style_classifier, style_tokenizer, alpha=0.5, *args, **kwargs):
        """
        Args:
            style_classifier: Detox classifier (e.g., shubhamkr/bert-style-classifier-detox)
            style_tokenizer: Corresponding tokenizer
            alpha: Weight for style reward term
        """
        super().__init__(*args, **kwargs)
        self.style_classifier = style_classifier
        self.style_tokenizer = style_tokenizer
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # --- Step 1: Standard supervised loss ---
        outputs = model(**inputs)
        ce_loss = outputs.loss

        processor = getattr(self, "processing_class", self.tokenizer)

        # --- Step 2: Generate predictions from the model ---
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=50
        )
        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Handle empty generations safely
        if not preds or all(len(p.strip()) == 0 for p in preds):
            return (ce_loss, outputs) if return_outputs else ce_loss

        # --- Step 3: Compute detox vs toxic classification ---
        style_inputs = self.style_tokenizer(
            preds,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            logits = self.style_classifier(**style_inputs).logits
            preds_cls = torch.argmax(logits, dim=-1)  # 0 = DETOX, 1 = TOXIC

        # --- Step 4: Convert classification into discrete rewards ---
        # DETOX → +1 ; TOXIC → −1
        rewards = torch.where(preds_cls == 0, 1.0, -1.0).to(model.device)

        # --- Step 5: Normalize rewards for stability ---
        std = rewards.std()
        if torch.isnan(std) or std < 1e-6:
            norm_rewards = rewards - rewards.mean()
        else:
            norm_rewards = (rewards - rewards.mean()) / (std + 1e-8)

        # --- Step 6: Combine cross-entropy loss and style reward ---
        # Subtract reward because higher reward → lower total loss
        total_loss = (1 - self.alpha) * ce_loss - self.alpha * norm_rewards.mean().detach()

        return (total_loss, outputs) if return_outputs else total_loss


##

## Full Finetuning of BART

In [ ]:
import shutil
import os
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import argparse
import torch
import math
import sacrebleu
from tqdm import tqdm
from scipy.stats.mstats import gmean
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    BartConfig
)
from huggingface_hub import HfApi, create_repo

def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)


def tokenize_datasets(df, src_col, trg_col, tokenizer, max_length):
    src_encodings = tokenizer(
        df[src_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    trg_encodings = tokenizer(
        df[trg_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    dataset = CreateDataset(src_encodings, trg_encodings)
    return dataset

def train_model(model, train_dataset, dev_dataset, tokenizer, args):
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    trainer = StyleRewardTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        style_classifier=style_classifier,
        style_tokenizer=style_tokenizer,
        alpha=0.5,
    )
    trainer.processing_class = tokenizer
    trainer.tokenizer = tokenizer

    trainer.train()
    return model

def generate_predictions(model, test_data, src_col, tokenizer, output_file):
    predictions = []
    for idx in range(len(test_data[src_col])):
        src = test_data[src_col].values[idx]
        src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
        generated_ids = model.generate(src_tknz["input_ids"].cuda(), max_length=args.max_length)
        prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(prediction)

    with open(output_file, 'wb') as f:
        pickle.dump(remove_prefix(predictions, 'neutral: '), f)

def generate_synthetic_data(model, unparallel_data, src_col, tokenizer):
    print(f"Generating synthetic train data from unparallel data...")
    predictions = []
    for idx in range(len(unparallel_data[src_col])):
        src = unparallel_data[src_col].values[idx]
        src = "toxic: " + src;
        src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
        generated_ids = model.generate(src_tknz["input_ids"].cuda(), max_length=args.max_length)
        prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(prediction)

    data_pairs = pd.DataFrame({
            'toxic': unparallel_data[src_col].values.tolist(),
            'neutral': remove_prefix(predictions, 'neutral: ')
    })
    return data_pairs


def evaluate_transfer(model, tokenizer, device, src, pred, target_label=None, threshold=0.9):
    """
    Evaluate the detoxification of 'pred' using the detox classifier.
    Returns:
        style_prob: probability if classified as DETOX, else 0
        label: predicted label ("DETOX" or "TOXIC")
        score: raw confidence score for predicted label
    """

    # Tokenize the prediction
    inputs = tokenizer(
        pred,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    # Run the classifier
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_label = torch.argmax(probs).item()
        confidence = probs[pred_label].item()

    # Assuming model labels:
    # 0 → DETOX
    # 1 → TOXIC
    if pred_label == 0:
        style_prob = confidence   # probability for DETOX
        label = "neutral"
    else:
        style_prob = 0.0          # not detoxified
        label = "toxic"

    return style_prob, label, confidence


def compute_scores_for_synthetic_pairs(sources, preds, evaluator, threshold=0.9):
    """
    Computes NLI-based style probability, BLEU, and semantic similarity for each pair.
    Returns: list of dicts [{src, pred, style_prob, bleu, sim, nli_label, nli_score}]
    """
    print(f"Computing scores for synthetic pairs using NLI evaluator...")
    sim_model = evaluator.sim_model
    out = []

    for src, pred in tqdm(zip(sources, preds), total=len(preds), desc="Scoring pairs"):
        style_prob, nli_label, nli_score = evaluate_transfer(evaluator.detox_model, evaluator.detox_tokenizer, evaluator.device, src, pred, threshold)

        bleu_vs_src = sacrebleu.sentence_bleu(pred, [src]).score / 100.0  # normalized 0–1

        emb = sim_model.encode([src, pred])
        cos_sim = float(np.dot(emb[0], emb[1]) / (np.linalg.norm(emb[0]) * np.linalg.norm(emb[1]) + 1e-9))
        sim_norm = (cos_sim + 1.0) / 2.0  # normalize 0–1

        out.append({
            'src': src,
            'pred': pred,
            'style_prob': float(style_prob),
            'bleu': float(bleu_vs_src),
            'sim': float(sim_norm),
            'nli_label': nli_label,
            'nli_score': float(nli_score)
        })

    print(f"Computed scores for {len(out)} synthetic pairs.")
    return out


def compute_filtered_score(item):
    """
    New scoring: total = style_prob * average(bleu, sim)
    """
    style_prob = item['style_prob']
    bleu = item['bleu']
    sim = item['sim']

    total_score = style_prob * bleu
    return float(total_score)


def add_scores_and_select(scored_list, top_k, high_thresh=0.9, low_thresh=0.7):
    """
    scored_list: list of dicts with keys ['style_prob', 'bleu', 'sim']
    top_k: number of examples to finally keep
    high_thresh: first (strict) threshold
    low_thresh: fallback threshold if too few examples above high_thresh

    Returns selected list sorted by score desc.
    """
    # Compute scores for all examples
    for it in scored_list:
        it['score'] = compute_filtered_score(it)

    # Sort descending by score
    scored_list.sort(key=lambda x: x['score'], reverse=True)
    high_selected = [x for x in scored_list if x['score'] >= 0.5 and x['score'] <= 0.95]

    if len(high_selected) >= top_k:
        return high_selected[:top_k]

    # 2️⃣ Not enough high-score examples → relax threshold
    low_selected = [x for x in scored_list if x['score'] >= low_thresh]

    # Return whichever is smaller (to avoid overshooting top_k)
    return low_selected[:min(len(low_selected), top_k)]



def progressive_self_training_round(
        model,
        tokenizer,
        unparallel_path,
        train_csv,
        dev_csv,
        output_dir,
        args,
        target_label,
        top_k,
        round_id,
        num_orig,
        seed_value=53,
        ):

    set_seed(seed_value)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n===== SELF-TRAINING ROUND {round_id} =====")

    # ---------- Stage 2: Generate ----------
    unparallel_df = remove_newline(pd.read_pickle(unparallel_path))
    unparallel_df = unparallel_df.sample(frac = 1, random_state = args.seed_value)
    unparallel_pred_df = generate_synthetic_data(model, unparallel_df, 'toxic', tokenizer)
    print("S2 complete!")

    # ---------- Stage 3: Score ----------
    evaluator = AutomaticEvaluator(seed_value)
    evaluator.set_seed()
    evaluator.load_models()
    sources = unparallel_pred_df["toxic"].tolist()
    preds = unparallel_pred_df["neutral"].tolist()
    scored = compute_scores_for_synthetic_pairs(sources, preds, evaluator)
    print("S3 complete!")

    # ---------- Stage 4: Select ----------
    selected = add_scores_and_select(scored, top_k=top_k)
    print(f"Selected top {len(selected)} pseudo pairs out of {len(scored)}")
    print("S4 complete!")

    # ---------- Stage 5: Merge -----------
    train_df = pd.read_csv(train_csv)
    total_synthetic_cap = 2000
    total_original_train =  1520
    sampled_train_df = train_df.sample(n=num_orig, random_state=seed_value)
    print(f"Selected {num_orig} original samples out of {total_original_train} for round {round_id}")
    print("S5 complete!")

    # Build pseudo DataFrame
    pseudo_df = pd.DataFrame({
      "neutral": [x["pred"] for x in selected],
      "toxic": [x["src"] for x in selected]
    })

    # Combine sampled original + synthetic
    augmented_train_df = pd.concat([sampled_train_df, pseudo_df], ignore_index=True)
    print(f"Augmented train data size: {len(augmented_train_df)}")


    # ----------- Stage 6: Fine-tune -----------
    print('Finetuning with the augmented data...')
    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '
    augmented_train_df["toxic"] = augmented_train_df["toxic"].apply(lambda x: pos_prompt+x)
    augmented_train_df["neutral"] = augmented_train_df["neutral"].apply(lambda x: neg_prompt+x)
    dev_df = remove_newline(pd.read_csv(dev_csv))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)
    dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt+x)
    dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt+x)
    augmented_train_dataset = tokenize_datasets(augmented_train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    training_args = Seq2SeqTrainingArguments(
      output_dir=output_dir,
      eval_strategy="epoch",
      learning_rate = 5e-5 if round_id == 4 else 1e-5,
      per_device_train_batch_size=args.batch_size,
      per_device_eval_batch_size=args.batch_size,
      weight_decay=args.weight_decay,
      save_total_limit=1,
      save_strategy='epoch',
      load_best_model_at_end=True,
      num_train_epochs = 4 if round_id == 4 else 2,
      predict_with_generate=True,
      fp16=True
    )

    # Train the model
    model = train_model(model, augmented_train_dataset, dev_dataset, tokenizer, training_args)
    print("S6 complete!")
    return model


class CreateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])
        return item

    def __len__(self):
        return len(self.labels['input_ids'])

def write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, filepath):
    with open(filepath, 'w') as file:
        file.write('Accuracy: ' + str(acc) + '\n')
        file.write('Bleu with Source: ' + str(bleu_withsrc) + '\n')
        file.write('Bleu with Target: ' + str(bleu_withtrg) + '\n')
        file.write('Similarity: ' + str(sim) + '\n')

    print('\n\n\n---------RESULTS---------\n')
    print('Accuracy: ' + str(acc) + '\n')
    print('Bleu with Source: ' + str(bleu_withsrc) + '\n')
    print('Similarity: ' + str(sim) + '\n')


def remove_prefix(strings, prefix):
    return [string.replace(prefix, '') for string in strings]

def remove_newline(df):
    df = df.apply(lambda x: x.str.replace('\n', ''))
    return df

def main(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '
    if args.prompt_enabled:

        train_df["toxic"] = train_df["toxic"].apply(lambda x: pos_prompt+x)
        train_df["neutral"] = train_df["neutral"].apply(lambda x: neg_prompt+x)

        dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt+x)
        dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt+x)

        test_df["toxic"] = test_df["toxic"].apply(lambda x: pos_prompt+x)
        test_df["neutral"] = test_df["neutral"].apply(lambda x: neg_prompt+x)

    with open('./output/detox/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/detox/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    tokenizer = BartTokenizer.from_pretrained(args.model_name)
    train_dataset = tokenize_datasets(train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Load the BART model
    model = BartForConditionalGeneration.from_pretrained(args.model_name)

    config = BartConfig.from_pretrained(args.model_name)

    config.dropout = 0.15

    config.attention_dropout = 0.05
    config.activation_dropout = 0.05

    config.label_smoothing_factor = 0.05

    model.config = config

    # Define a directory to save the model
    output_directory = "./results"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    training_args = Seq2SeqTrainingArguments(
      output_dir=output_directory,
      eval_strategy="epoch",
      learning_rate=args.learning_rate,
      per_device_train_batch_size=args.batch_size,
      per_device_eval_batch_size=args.batch_size,
      weight_decay=args.weight_decay,
      save_total_limit=1,
      save_strategy='epoch',
      load_best_model_at_end=True,
      num_train_epochs= 3,
      predict_with_generate=True,
      fp16=True
    )

    # Train the model
    model = train_model(model, train_dataset, dev_dataset, tokenizer, training_args)

    top_k = 500
    maxtopk = 2000
    num_orig = 300

    for round_id in range(1, 5):
      model = progressive_self_training_round(
            model=model,
            tokenizer=tokenizer,
            unparallel_path=args.unparallel_data,
            train_csv=args.train_file,
            dev_csv=args.dev_file,
            output_dir='./results/selftrain',
            args = args,
            target_label="neutral",
            top_k=top_k,
            round_id=round_id,
            num_orig=num_orig
        )
      top_k = min(top_k*2, maxtopk)
      num_orig += 300
      print(f"Round {round_id} completed.")


    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file)
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()


    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)

    repo_name = "bart-tst-llrpd-detox-vf"
    username = "shubhamkr"

    # Create the repo on Hugging Face (if not exists)
    create_repo(f"{username}/{repo_name}", exist_ok=True)

    # Save locally first
    save_dir = f"content/genai_proj_low_resource_tst/{repo_name}"
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    # Then push to Hub
    model.push_to_hub(f"{username}/{repo_name}")
    tokenizer.push_to_hub(f"{username}/{repo_name}")

## Start


In [ ]:
%cd /content/genai_proj_low_resource_tst


from types import SimpleNamespace
import os
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("/content/genai_proj_low_resource_tst/output/detox", exist_ok=True)


if __name__ == "__main__":
    args = SimpleNamespace(
        seed_value=53,
        model_name="facebook/bart-base",
        max_length=128,
        batch_size=8,
        learning_rate=5e-5,
        weight_decay=0.0,
        num_train_epochs=3,
        dropout=0.0,
        attention_dropout=0.0,
        activation_dropout=0.0,
        label_smoothing_factor=0.0,

        train_file="./data/train.csv",
        dev_file="./data/dev.csv",
        test_file="./data/test.pkl",
        unparallel_data="./data/unparallel.pkl",

        pred_file="./output/detox/pred.pkl",
        src_file="./output/detox/src.pkl",
        ref_file="./output/detox/trg.pkl",
        output_file = "./output/detox/pred.pkl",
        synthetic_train_file="./data/synthetic_train.csv",
        eval_scores_file="./output/detox/scores.txt",
        prompt_enabled=True,
        src_col="toxic",
        trg_col="neutral",
        dev_size=0.1,
        train_pkl_path="./output/detox/train.pkl",
        task='toxic_to_neutral'
    )

# main(args)

## Parameter Efficient Finetuning with LoRA

In [ ]:
%pip install transformers datasets sacrebleu peft accelerate bitsandbytes -q


In [ ]:
import shutil
import os
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import argparse
import torch
import math
import sacrebleu
from tqdm import tqdm
from scipy.stats.mstats import gmean
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    BartConfig
)
from huggingface_hub import HfApi, create_repo

# PEFT / LoRA
from peft import LoraConfig, get_peft_model, TaskType


def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)


def tokenize_datasets(df, src_col, trg_col, tokenizer, max_length):
    src_encodings = tokenizer(
        df[src_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    trg_encodings = tokenizer(
        df[trg_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    dataset = CreateDataset(src_encodings, trg_encodings)
    return dataset


def prepare_peft_model(model, args):
    if not args.lora_enable:
        return model, None

    target_modules = args.lora_target_modules.split(',') if args.lora_target_modules else None
    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        target_modules=target_modules,
        lora_dropout=args.lora_dropout,
        bias="all",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )

    model = get_peft_model(model, lora_config)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"PEFT/LoRA enabled — total params: {total_params:,}, trainable: {trainable_params:,}")
    return model, lora_config


def train_model(model, train_dataset, dev_dataset, tokenizer, args):
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        eval_strategy=args.eval_strategy,
        learning_rate=args.learning_rate,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        weight_decay=args.weight_decay,
        save_total_limit=args.save_total_limit,
        save_strategy=args.save_strategy,
        load_best_model_at_end=args.load_best_model_at_end,
        num_train_epochs=args.num_train_epochs,
        predict_with_generate=True,
        fp16=args.fp16 and torch.cuda.is_available(),
        remove_unused_columns=False,
        push_to_hub=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    trainer.train()
    return model

def generate_predictions(model, test_data, src_col, tokenizer, output_file, args):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    with torch.no_grad():
        for idx in range(len(test_data[src_col])):
            src = test_data[src_col].values[idx]
            src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
            input_ids = src_tknz["input_ids"].to(device)
            attention_mask = src_tknz.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)

            generated_ids = model.generate(input_ids=input_ids,
                                           attention_mask=attention_mask,
                                           max_length=args.max_length)
            prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            predictions.append(prediction)

    with open(output_file, 'wb') as f:
        pickle.dump(remove_prefix(predictions, 'neutral: '), f)


def generate_synthetic_data(model, unparallel_data, src_col, tokenizer, args):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    with torch.no_grad():
        for idx in range(len(unparallel_data[src_col])):
            src = unparallel_data[src_col].values[idx]
            src = "toxic: " + src
            src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
            input_ids = src_tknz["input_ids"].to(device)
            attention_mask = src_tknz.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)

            generated_ids = model.generate(input_ids=input_ids,
                                           attention_mask=attention_mask,
                                           max_length=args.max_length)
            prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            predictions.append(prediction)

    data_pairs = pd.DataFrame({
            'toxic': unparallel_data[src_col].values.tolist(),
            'neutral': remove_prefix(predictions, 'neutral: ')
    })
    return data_pairs


def evaluate_transfer(model, tokenizer, device, src, pred, target_label=None, threshold=0.9):
    """
    Evaluate the detoxification of 'pred' using the detox classifier.
    Returns:
        style_prob: probability if classified as DETOX, else 0
        label: predicted label ("DETOX" or "TOXIC")
        score: raw confidence score for predicted label
    """

    # Tokenize the prediction
    inputs = tokenizer(
        pred,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    # Run the classifier
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_label = torch.argmax(probs).item()
        confidence = probs[pred_label].item()

    # Assuming model labels:
    # 0 → DETOX
    # 1 → TOXIC
    if pred_label == 0:
        style_prob = confidence   # probability for DETOX
        label = "neutral"
    else:
        style_prob = 0.0          # not detoxified
        label = "toxic"

    return style_prob, label, confidence


def compute_scores_for_synthetic_pairs(sources, preds, evaluator, threshold=0.9):
    """
    Computes NLI-based style probability, BLEU, and semantic similarity for each pair.
    Returns: list of dicts [{src, pred, style_prob, bleu, sim, nli_label, nli_score}]
    """
    print(f"Computing scores for synthetic pairs using NLI evaluator...")
    sim_model = evaluator.sim_model
    out = []

    for src, pred in tqdm(zip(sources, preds), total=len(preds), desc="Scoring pairs"):
        style_prob, nli_label, nli_score = evaluate_transfer(evaluator.detox_model, evaluator.detox_tokenizer, evaluator.device, src, pred, threshold)

        bleu_vs_src = sacrebleu.sentence_bleu(pred, [src]).score / 100.0  # normalized 0–1

        emb = sim_model.encode([src, pred])
        cos_sim = float(np.dot(emb[0], emb[1]) / (np.linalg.norm(emb[0]) * np.linalg.norm(emb[1]) + 1e-9))
        sim_norm = (cos_sim + 1.0) / 2.0  # normalize 0–1

        out.append({
            'src': src,
            'pred': pred,
            'style_prob': float(style_prob),
            'bleu': float(bleu_vs_src),
            'sim': float(sim_norm),
            'nli_label': nli_label,
            'nli_score': float(nli_score)
        })

    print(f"Computed scores for {len(out)} synthetic pairs.")
    return out


def compute_filtered_score(item):
    """
    New scoring: total = style_prob * average(bleu, sim)
    """
    style_prob = item['style_prob']
    bleu = item['bleu']
    sim = item['sim']

    total_score = style_prob * bleu
    return float(total_score)


def add_scores_and_select(scored_list, top_k, high_thresh=0.9, low_thresh=0.7):
    """
    scored_list: list of dicts with keys ['style_prob', 'bleu', 'sim']
    top_k: number of examples to finally keep
    high_thresh: first (strict) threshold
    low_thresh: fallback threshold if too few examples above high_thresh

    Returns selected list sorted by score desc.
    """
    # Compute scores for all examples
    for it in scored_list:
        it['score'] = compute_filtered_score(it)

    # Sort descending by score
    scored_list.sort(key=lambda x: x['score'], reverse=True)
    high_selected = [x for x in scored_list if x['score'] >= 0.5 and x['score'] <= 0.95]

    if len(high_selected) >= top_k:
        return high_selected[:top_k]

    low_selected = [x for x in scored_list if x['score'] >= low_thresh]

    # Return whichever is smaller (to avoid overshooting top_k)
    return low_selected[:min(len(low_selected), top_k)]


def progressive_self_training_round(
        model,
        tokenizer,
        unparallel_path,
        train_csv,
        dev_csv,
        output_dir,
        args,
        target_label,
        top_k,
        round_id,
        num_orig,
        seed_value=53,
):
    """
    Run one round of progressive self-training using the LoRA-enabled model.
    """
    set_seed(seed_value)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n===== SELF-TRAINING ROUND {round_id} =====")

    # ---------- Stage 2: Generate ----------
    unparallel_df = remove_newline(pd.read_pickle(unparallel_path))
    unparallel_df = unparallel_df.sample(frac=1, random_state=args.seed_value)
    unparallel_pred_df = generate_synthetic_data(model, unparallel_df, 'toxic', tokenizer, args)
    print("S2 complete!")

    # ---------- Stage 3: Score ----------
    evaluator = AutomaticEvaluator(seed_value)
    evaluator.set_seed()
    evaluator.load_models()
    sources = unparallel_pred_df["toxic"].tolist()
    preds = unparallel_pred_df["neutral"].tolist()
    scored = compute_scores_for_synthetic_pairs(sources, preds, evaluator)
    print("S3 complete!")

    # ---------- Stage 4: Select ----------
    selected = add_scores_and_select(scored, top_k=top_k)
    print(f"Selected top {len(selected)} pseudo pairs out of {len(scored)}")
    print("S4 complete!")

    # ---------- Stage 5: Merge ----------
    train_df = pd.read_csv(train_csv)
    sampled_train_df = train_df.sample(n=num_orig, random_state=seed_value)
    pseudo_df = pd.DataFrame({
        "neutral": [x["pred"] for x in selected],
        "toxic": [x["src"] for x in selected],
    })
    augmented_train_df = pd.concat([sampled_train_df, pseudo_df], ignore_index=True)
    print(f"Augmented train data size: {len(augmented_train_df)}")

    # ---------- Stage 6: Fine-tune ----------
    print('Finetuning with the augmented data...')
    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '

    augmented_train_df["toxic"] = augmented_train_df["toxic"].apply(lambda x: pos_prompt + x)
    augmented_train_df["neutral"] = augmented_train_df["neutral"].apply(lambda x: neg_prompt + x)

    dev_df = remove_newline(pd.read_csv(dev_csv))
    dev_df = dev_df.sample(frac=1, random_state=args.seed_value)
    dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt + x)
    dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt + x)

    augmented_train_dataset = tokenize_datasets(augmented_train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Reuse existing PEFT-aware train_model()
    model = train_model(model, augmented_train_dataset, dev_dataset, tokenizer, args)
    print("S6 complete!")

    return model



class CreateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])
        return item

    def __len__(self):
        return len(self.labels['input_ids'])


def write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, filepath):
    with open(filepath, 'w') as file:
        file.write('Accuracy: ' + str(acc) + '\n')
        file.write('Bleu with Source: ' + str(bleu_withsrc) + '\n')
        file.write('Bleu with Target: ' + str(bleu_withtrg) + '\n')
        file.write('Similarity: ' + str(sim) + '\n')

    print('\n\n\n---------RESULTS---------\n')
    print('Accuracy: ' + str(acc) + '\n')
    print('Bleu with Source: ' + str(bleu_withsrc) + '\n')
    print('Similarity: ' + str(sim) + '\n')


def remove_prefix(strings, prefix):
    return [string.replace(prefix, '') for string in strings]


def remove_newline(df):
    df = df.apply(lambda x: x.str.replace('\n', ''))
    return df


def main(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '
    if args.prompt_enabled:

        train_df["toxic"] = train_df["toxic"].apply(lambda x: pos_prompt+x)
        train_df["neutral"] = train_df["neutral"].apply(lambda x: neg_prompt+x)

        dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt+x)
        dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt+x)

        test_df["toxic"] = test_df["toxic"].apply(lambda x: pos_prompt+x)
        test_df["neutral"] = test_df["neutral"].apply(lambda x: neg_prompt+x)

    with open('./output/detox/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/detox/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    tokenizer = BartTokenizer.from_pretrained(args.model_name)

    train_dataset = tokenize_datasets(train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Load the BART model
    model = BartForConditionalGeneration.from_pretrained(args.model_name)

    config = BartConfig.from_pretrained(args.model_name)
    config.dropout = 0.15
    config.attention_dropout = 0.05
    config.activation_dropout = 0.05
    config.label_smoothing_factor = 0.05
    model.config = config

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Prepare PEFT/LoRA if enabled
    model, lora_config = prepare_peft_model(model, args)

    # Train the model
    model = train_model(model, train_dataset, dev_dataset, tokenizer, args)

    # ---------- Progressive Self-Training ----------
    top_k = 500
    maxtopk = 2000
    num_orig = 300

    for round_id in range(1, 5):
        model = progressive_self_training_round(
            model=model,
            tokenizer=tokenizer,
            unparallel_path=args.unparallel_data,
            train_csv=args.train_file,
            dev_csv=args.dev_file,
            output_dir=os.path.join(args.output_dir, f"selftrain_round{round_id}"),
            args=args,
            target_label="neutral",
            top_k=top_k,
            round_id=round_id,
            num_orig=num_orig
        )
        top_k = min(top_k * 2, maxtopk)
        num_orig += 300
        print(f"Round {round_id} completed.")


    # Save PEFT adapter + tokenizer
    save_dir = os.path.join(args.output_dir, "peft_adapter")
    os.makedirs(save_dir, exist_ok=True)
    # If model is a PEFT model, use save_pretrained to save adapter only
    try:
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Saved PEFT adapter + tokenizer to {save_dir}")
    except Exception as e:
        print("Warning: failed to use model.save_pretrained() — saving full model instead. Error:", e)
        model.save_pretrained(os.path.join(args.output_dir, "full_model"))
        tokenizer.save_pretrained(os.path.join(args.output_dir, "full_model"))

    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file, args)
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)


    repo_name = "bart-tst-llrpd-detox-peft-v1"
    username = "shubhamkr"

    # Create the repo on Hugging Face (if not exists)
    create_repo(f"{username}/{repo_name}", exist_ok=True)

    # Then push to Hub
    model.push_to_hub(f"{username}/{repo_name}", include_adapter=True)
    tokenizer.push_to_hub(f"{username}/{repo_name}")


In [ ]:
# cd into your project
%cd /content/genai_proj_low_resource_tst/

from types import SimpleNamespace
import os
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("/content/genai_proj_low_resource_tst/output/detox", exist_ok=True)

if __name__ == "__main__":
    args = SimpleNamespace(
        seed_value=53,
        model_name="facebook/bart-base",
        max_length=128,
        batch_size=8,
        learning_rate=1e-4,
        weight_decay=0.0,
        num_train_epochs=3,
        dropout=0.05,
        attention_dropout=0.0,
        activation_dropout=0.0,
        label_smoothing_factor=0.0,

        # data paths (keep your original structure)
        train_file="./data/train.csv",
        dev_file="./data/dev.csv",
        test_file="./data/test.pkl",
        unparallel_data="./data/unparallel.pkl",

        # outputs & evaluation
        pred_file="./output/detox/pred.pkl",
        src_file="./output/detox/src.pkl",
        ref_file="./output/detox/trg.pkl",
        output_dir="./output/detox/results",
        synthetic_train_file="./data/synthetic_train.csv",
        eval_scores_file="./output/detox/scores.txt",

        prompt_enabled=True,
        src_col="toxic",
        trg_col="neutral",
        dev_size=0.1,
        train_pkl_path="./output/detox/train.pkl",
        task='toxic_to_neutral',

        # PEFT / LoRA flags
        lora_enable=True,
        lora_r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        lora_target_modules="q_proj,k_proj,v_proj,o_proj,fc1,fc2",

        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        fp16=False,
    )

    # main(args)


## Evaluation of Fully Trained Model

In [ ]:
def check_evaluation(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '
    if args.prompt_enabled:

        train_df["toxic"] = train_df["toxic"].apply(lambda x: pos_prompt+x)
        train_df["neutral"] = train_df["neutral"].apply(lambda x: neg_prompt+x)

        dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt+x)
        dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt+x)

        test_df["toxic"] = test_df["toxic"].apply(lambda x: pos_prompt+x)
        test_df["neutral"] = test_df["neutral"].apply(lambda x: neg_prompt+x)

    with open('./output/detox/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/detox/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    model_name = "shubhamkr/bart-tst-llrpd-detox"

    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    train_dataset = tokenize_datasets(train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file)
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()


    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)


check_evaluation(args)


## Evaluation of PEFT Trained Model

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
from peft import PeftModel
import torch


def check_evaluation(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'neutral: '
    pos_prompt = 'toxic: '
    if args.prompt_enabled:

        train_df["toxic"] = train_df["toxic"].apply(lambda x: pos_prompt+x)
        train_df["neutral"] = train_df["neutral"].apply(lambda x: neg_prompt+x)

        dev_df["toxic"] = dev_df["toxic"].apply(lambda x: pos_prompt+x)
        dev_df["neutral"] = dev_df["neutral"].apply(lambda x: neg_prompt+x)

        test_df["toxic"] = test_df["toxic"].apply(lambda x: pos_prompt+x)
        test_df["neutral"] = test_df["neutral"].apply(lambda x: neg_prompt+x)

    with open('./output/detox/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/detox/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    model_name = "facebook/bart-base"
    peft_model_name = "shubhamkr/bart-tst-llrpd-detox-peft"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = BartTokenizer.from_pretrained(peft_model_name)
    base_model = BartForConditionalGeneration.from_pretrained(model_name)
    model = PeftModel.from_pretrained(base_model, peft_model_name)
    model = model.to(device)
    model.eval()

    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file, args)
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)


check_evaluation(args)